In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [2]:
class DeepRNNNet(nn.Module):
    def __init__(self, input_size=12289, hidden_size_1=512, hidden_size_2=128,
                 output_size=2, num_layers=1):  # num_layers теперь равен 2
        super(DeepRNNNet, self).__init__()
        self.num_layers = num_layers
        self.hidden_size = hidden_size_1
        self.rnn = nn.RNN(input_size, hidden_size_1, num_layers, batch_first=True)
        self.fc1 = nn.Linear(hidden_size_1, hidden_size_2)
        self.fc2 = nn.Linear(hidden_size_2, output_size)
        self.relu = nn.ReLU()
        self.tanh = nn.Tanh()

    def forward(self, input_data):
        h0 = torch.zeros(self.num_layers, 1, self.hidden_size).to(input_data.device)

        out, hn = self.rnn(input_data, h0)

        out = self.fc1(out[:, -1, :])
        out = self.relu(out)
        out = self.fc2(out)
        out = self.tanh(out)
        return out

In [3]:
# Подготовка данных (пример)
X = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_road_rnn.csv')
S = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_speed_rnn.csv')
y = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\y_train_wheel_rnn.csv')

In [4]:
X_train = torch.tensor(X.values, dtype=torch.float32).to(device='cuda')
S_train = torch.tensor(S.values, dtype=torch.float32).to(device='cuda')
y_tensor = torch.tensor(y.values, dtype=torch.float32).to(device='cuda')

In [5]:
print(len(X_train))
print(len(S_train))
print(len(y_tensor))

12781
12781
12781


In [6]:
X_tensor = torch.cat((X_train, S_train), dim=1)
print(X_tensor[0])
print(X_tensor[0].size())

tensor([  0.,   0.,   0.,  ...,   0.,   0., 104.], device='cuda:0')
torch.Size([12289])


In [7]:
print(y_tensor[0])
print(y_tensor[0].size())

tensor([-0.0071, -0.0060], device='cuda:0')
torch.Size([2])


In [8]:
dataset = TensorDataset(X_tensor, y_tensor)
train_loader = DataLoader(dataset, batch_size=1024, shuffle=False)

In [9]:
from torch.optim import lr_scheduler
model = DeepRNNNet().cuda()

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0000000001)
scheduler = lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

# Обучение модели
num_epochs = 1
for epoch in range(num_epochs):
    total_loss = 0.0  # Для вычисления средней потери за эпоху
    for X, y in train_loader:
        X = X.unsqueeze(0)
        X, y = X.cuda(), y.cuda()  # Убедитесь, что данные и метки отправлены на GPU

        optimizer.zero_grad()  # Обнуляем градиенты
        output = model(X)  # Прямой проход
        loss = criterion(output, y)  # Вычисление потери
        loss.backward()  # Обратное распространение
        optimizer.step()  # Шаг оптимизации

        total_loss += loss.item() * X.size(0)  # Умножаем потерю на размер пакета для корректного усреднения

    average_loss = total_loss / len(train_loader.dataset)  # Средняя потеря за эпоху
    print(f'Epoch [{epoch + 1}/{num_epochs}], Average Loss: {average_loss:.9f}')

C:\PycharmProjects\ETS_Autopilot\venv_3_11\Lib\site-packages\torch\nn\modules\loss.py:535: UserWarning: Using a target size (torch.Size([1024, 2])) that is different to the input size (torch.Size([1, 2])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch [1/1], Average Loss: 0.000006043


C:\PycharmProjects\ETS_Autopilot\venv_3_11\Lib\site-packages\torch\nn\modules\loss.py:535: UserWarning: Using a target size (torch.Size([493, 2])) that is different to the input size (torch.Size([1, 2])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


In [11]:
torch.save(model.state_dict(), 'C:\PycharmProjects\ETS_Autopilot\static\weight_model\weight_wheel_nn_rnn_version_1.pth')